# 🤖 Classification de Sentiments avec Hugging Face

Dans cette étape, nous allons utiliser un modèle de **Deep Learning pré-entraîné**. 
Au lieu de créer un modèle depuis zéro en lui apprenant que le mot "super" est positif, nous allons télécharger un modèle issu de la plateforme **Hugging Face** qui a déjà lu des millions de textes sur internet et qui "comprend" le français.

Nous utiliserons le modèle `nlptown/bert-base-multilingual-uncased-sentiment` qui note les textes de 1 à 5 étoiles.

In [1]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import classification_report

# 1. Téléchargement et initialisation du modèle
# (La première exécution télécharge le modèle, environ 600 Mo)
analyseur = pipeline(
    task="sentiment-analysis", 
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

print("✅ Modèle chargé avec succès !")

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

C:\Users\jovyb\.gemini\antigravity\scratch\sentiment_analysis\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jovyb\.cache\huggingface\hub\models--nlptown--bert-base-multilingual-uncased-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\jovyb\.gemini\antigravity\scratch\sentiment_

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ Modèle chargé avec succès !


## 1. Test Rapide
Essayons notre modèle sur quelques phrases inventées.

In [2]:
phrases_test = [
    "Ce produit est absolument génial, je l'adore !",
    "C'est correct, sans plus. Fait le travail.",
    "Pire achat de ma vie, le service client est une catastrophe."
]

for phrase in phrases_test:
    resultat = analyseur(phrase)[0]
    print(f"Texte : '{phrase}'")
    print(f"-> Prédiction : {resultat['label']} (Confiance : {resultat['score']*100:.1f}%)\n")

Texte : 'Ce produit est absolument génial, je l'adore !'
-> Prédiction : 5 stars (Confiance : 89.3%)

Texte : 'C'est correct, sans plus. Fait le travail.'
-> Prédiction : 3 stars (Confiance : 72.0%)

Texte : 'Pire achat de ma vie, le service client est une catastrophe.'
-> Prédiction : 1 star (Confiance : 93.1%)



## 2. Prédiction sur notre Jeu de Données

Nous allons appliquer ce modèle à nos 700 avis générés lors de l'étape précédente. 
*(Note : Sur un processeur classique (CPU), cela peut prendre une ou deux minutes car le modèle lit de vraies phrases de manière contextuelle).* 

Pour simplifier, nous allons convertir la prédiction du modèle (de 1 à 5 étoiles) en nos 3 catégories métier : Positif, Neutre, Négatif.

In [3]:
df = pd.read_csv('../data/avis_clients.csv')

# Fonction pour traduire les étoiles en Sentiment
def etoiles_vers_sentiment(label):
    etoiles = int(label.split()[0]) # Récupère le chiffre dans "5 stars"
    if etoiles >= 4:
        return 'Positif'
    elif etoiles == 3:
        return 'Neutre'
    else:
        return 'Negatif'

print("L'IA lit les 700 avis... Veuillez patienter ⏳")

# On applique le modèle sur tous les textes originaux (pas besoin de nettoyage extrême avec BERT !)
# On extrait le 'label' de chaque résultat, puis on le convertit.
df['Prediction_IA'] = [etoiles_vers_sentiment(res['label']) for res in analyseur(df['Texte'].tolist())]

print("✅ Terminé !")
display(df[['Texte', 'Sentiment_Reel', 'Prediction_IA']].sample(5))

L'IA lit les 700 avis... Veuillez patienter ⏳


✅ Terminé !


,Texte,Sentiment_Reel,Prediction_IA
452,La livraison a pris un mois et le colis était ...,Negatif,Negatif
387,Le produit reçu ne correspond pas à la photo d...,Negatif,Negatif
123,Le produit reçu ne correspond pas à la photo d...,Negatif,Negatif
589,"Conforme aux attentes, ni bon ni mauvais.",Neutre,Positif
64,"Produit correct, fait le job mais sans plus.",Neutre,Neutre


## 3. Évaluation des Performances

Le modèle est-il bon ? Comparons ses prédictions avec le "Sentiment_Reel" que nous connaissons.

In [4]:
print(classification_report(df['Sentiment_Reel'], df['Prediction_IA']))

              precision    recall  f1-score   support

     Negatif       1.00      1.00      1.00       250
      Neutre       1.00      0.62      0.77       150
     Positif       0.84      1.00      0.91       300

    accuracy                           0.92       700
   macro avg       0.95      0.87      0.89       700
weighted avg       0.93      0.92      0.91       700



**Conclusion** :
Les modèles de type **Transformer** (comme BERT) sont incroyablement performants pour ce genre de tâche, et ce, **sans aucun entraînement de notre part** (on appelle ça du *Zero-Shot* ou de l'inférence pré-entraînée).

Nous avons notre moteur d'Intelligence Artificielle prêt pour notre tableau de bord !